In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 0: Create Workflow
# MAGIC Creates a Databricks Workflow that chains all four pipeline notebooks
# MAGIC as dependent tasks using the Databricks SDK.
# MAGIC
# MAGIC **Run this notebook once to register the workflow.**
# MAGIC After that, trigger it from the Workflows UI or via the SDK run call at the bottom.

# COMMAND ----------

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import (
    Task,
    NotebookTask,
    TaskDependency,
)

w = WorkspaceClient()

# COMMAND ----------

# Derive the base path from this notebook's current location.
# All notebooks must be in the same workspace folder.

current_notebook_path = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

base_folder = current_notebook_path.rsplit("/", 1)[0]

notebook_world_state   = f"{base_folder}/notebook1_world_state"
notebook_agent_loop    = f"{base_folder}/notebook2_agent_loop_llm"
notebook_investigation = f"{base_folder}/notebook3_investigation"
notebook_dashboard     = f"{base_folder}/notebook4_lakeview"

print("Resolved notebook paths:")
print(f"  Task 1: {notebook_world_state}")
print(f"  Task 2: {notebook_agent_loop}")
print(f"  Task 3: {notebook_investigation}")
print(f"  Task 4: {notebook_dashboard}")

# COMMAND ----------

# Delete any existing workflow with the same name to allow clean reruns

existing_jobs = list(w.jobs.list(name="SupplyAgent: Point-in-Time Correctness Pipeline"))

if existing_jobs:
    for existing_job in existing_jobs:
        w.jobs.delete(job_id=existing_job.job_id)
        print(f"Deleted existing workflow: job_id={existing_job.job_id}")

# COMMAND ----------

# Create the workflow with four sequential tasks.
# Each task depends on the previous one completing successfully.
# Serverless compute is used — no cluster configuration needed on Free Edition.

created_job = w.jobs.create(
    name="SupplyAgent: Point-in-Time Correctness Pipeline",
    tasks=[
        Task(
            task_key="world_state_simulation",
            description=(
                "Generates synthetic IoT sensor and weather API readings. "
                "Injects a corrupted reading window and self-correction. "
                "Writes three Delta versions tagged with userMetadata."
            ),
            notebook_task=NotebookTask(
                notebook_path=notebook_world_state
            )
        ),
        Task(
            task_key="agent_decision_loop",
            description=(
                "Runs SupplyAgent across three simulated decision times. "
                "LLM reads world_state at each event_timestamp and writes "
                "decisions to agent_memory with bi-temporal timestamps."
            ),
            depends_on=[
                TaskDependency(task_key="world_state_simulation")
            ],
            notebook_task=NotebookTask(
                notebook_path=notebook_agent_loop
            )
        ),
        Task(
            task_key="point_in_time_investigation",
            description=(
                "Reconstructs the exact context the agent operated on at 08:14 "
                "using Delta Time Travel. Performs bi-temporal audit and OPTIMIZE."
            ),
            depends_on=[
                TaskDependency(task_key="agent_decision_loop")
            ],
            notebook_task=NotebookTask(
                notebook_path=notebook_investigation
            )
        ),
        Task(
            task_key="lakeview_dashboard",
            description=(
                "Creates and publishes the Lakeview AI/BI dashboard. "
                "Visualizes the sensor timeline, agent decisions, "
                "bi-temporal audit, and Delta commit history."
            ),
            depends_on=[
                TaskDependency(task_key="point_in_time_investigation")
            ],
            notebook_task=NotebookTask(
                notebook_path=notebook_dashboard
            )
        ),
    ]
)

print(f"Workflow created successfully.")
print(f"  Job ID : {created_job.job_id}")
print(f"  View it: {w.config.host}#job/{created_job.job_id}")

# COMMAND ----------

# Trigger an immediate run of the full pipeline

run_response = w.jobs.run_now(job_id=created_job.job_id)

print(f"Pipeline run triggered.")
print(f"  Run ID  : {run_response.run_id}")
print(f"  Track it: {w.config.host}#job/{created_job.job_id}/run/{run_response.run_id}")

In [0]:
spark.sql("SELECT current_catalog()").show()

In [0]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

# Current SupplyAgent dashboard
supply = w.lakeview.get(dashboard_id="01f15592577e1e63a9898cc8a370fd0c")
print("=== SUPPLY AGENT SPEC ===")
print(supply.serialized_dashboard)

print("\n\n")

# Dashboard created today — might have working chart config
today = w.lakeview.get(dashboard_id="01f1558477fc16df9897385eddc358b5")
print("=== TODAY DASHBOARD SPEC ===")
print(today.serialized_dashboard)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

DASHBOARD_NAME = "SupplyAgent: Point-in-Time Correctness Audit"

dashboard = next(
    d for d in w.lakeview.list()
    if d.display_name == DASHBOARD_NAME
)

# Fetch full dashboard with spec
full_dashboard = w.lakeview.get(dashboard_id=dashboard.dashboard_id)

print(full_dashboard.serialized_dashboard)